In [ ]:
%matplotlib inline

import _plotting as plot
import jax
from jax import numpy as jnp
from matplotlib import pyplot as plt

from xxm.core.align import align_procrustes as align_latent
from xxm.lds import PoissonLDS

In [ ]:
def build_lds_example(
    latent_dim=2,
    num_neurons=12,
    angle=0.15,
    radius=0.995,
    base_rate=1.0,
) -> PoissonLDS:
    """Build an LDS example with a damped rotation dynamics."""

    dynamics_matrix = radius * jnp.array(
        [
            [jnp.cos(angle), -jnp.sin(angle)],
            [jnp.sin(angle), jnp.cos(angle)],
        ]
    )

    preferred_angles = jnp.linspace(
        0.0,
        2.0 * jnp.pi,
        num_neurons,
        endpoint=False,
    )

    readout = 0.7 * jnp.stack(
        [
            jnp.cos(preferred_angles),
            jnp.sin(preferred_angles),
        ],
        axis=1,
    )

    return PoissonLDS.from_params(
        initial_mean=jnp.array([1.5, 0.0]),
        initial_covariance=0.05 * jnp.eye(latent_dim),
        dynamics_coefficients=dynamics_matrix,
        dynamics_bias=jnp.zeros(latent_dim),
        dynamics_covariance=0.015 * jnp.eye(latent_dim),
        emission_coefficients=readout,
        emission_bias=jnp.full((num_neurons,), jnp.log(base_rate)),
    )


true_model = build_lds_example()

true_latents, observations = true_model.sample(
    key=jax.random.key(0),
    num_steps=500,
)


def plot_data(latents, observations):

    _, axs = plt.subplots(
        ncols=2,
        figsize=(8, 3),
        constrained_layout=True,
        gridspec_kw={'width_ratios': [1, 2]},
    )

    ax = axs[0]

    plot.plot_traces_2d(ax, latents)

    ax = axs[1]
    plot.plot_traces_image(ax, observations)


plot_data(true_latents, observations)

In [ ]:
posterior, _ = true_model.infer(observations)

In [ ]:
def plot_inference(latents, posterior):
    _, axs = plt.subplots(
        ncols=2, gridspec_kw={'width_ratios': [1, 2]}, figsize=(12, 4)
    )

    ax = axs[0]
    plot.plot_traces_2d(ax, latents, color='k', label='true')
    plot.plot_traces_2d(
        ax, posterior.means, color='xkcd:apple green', label='inferred mean'
    )

    ax.legend(loc='upper right')

    ax = axs[1]

    std = jnp.sqrt(posterior.covariances[:, 0, 0])

    ax.plot(latents[:, 0], label='true', color='k')
    ax.plot(posterior.means[:, 0], label='posterior', color='xkcd:apple green')
    ax.fill_between(
        jnp.arange(len(latents)),
        posterior.means[:, 0] - 2 * std,
        posterior.means[:, 0] + 2 * std,
        alpha=0.2,
    )


plot_inference(true_latents, posterior)

In [ ]:
initial_models = PoissonLDS.from_pca_many(
    observations,
    latent_dim=2,
    covariance_floors=jnp.logspace(-5, -2, 4),
)

In [ ]:
fits = PoissonLDS.fit_many(
    initial_models,
    observations,
    num_iters=50,
)

best_idx = fits.best_index()

initial_model = initial_models[best_idx]
fitted_model = fits.get(best_idx)[0]

plot.plot_fit_progress_many(fits, best_idx)

In [ ]:
def align_model(learned_model, true_latents):

    posterior, _ = learned_model.infer(observations)

    inferred_latents = learned_model.latent_mean(posterior)

    alignment = align_latent(inferred_latents, true_latents)

    return learned_model.align(alignment)


initial_model = align_model(initial_model, true_latents)
fitted_model = align_model(fitted_model, true_latents)

initial_posterior, _ = initial_model.infer(observations)
initial_latent = initial_model.latent_mean(initial_posterior)

fitted_posterior, _ = fitted_model.infer(observations)
fitted_latent = fitted_model.latent_mean(fitted_posterior)

In [ ]:
def plot_latent_comparison(latents_true, initial_latent, fitted_latent):
    _, axes = plt.subplots(1, 3, figsize=(12, 4))

    ax = axes[0]
    plot.plot_traces_2d(ax, latents_true)
    ax.set(
        title='True latent',
        aspect='equal',
    )

    ax = axes[1]
    plot.plot_traces_2d(ax, initial_latent, color='xkcd:grey')
    ax.set(
        title='Initial posterior',
        aspect='equal',
    )

    ax = axes[2]
    plot.plot_traces_2d(ax, fitted_latent, color='xkcd:apple green')
    ax.set(
        title='Fitted posterior',
        aspect='equal',
    )


plot_latent_comparison(true_latents, initial_latent, fitted_latent)

In [ ]:
initial_prediction = initial_model.observation_mean(initial_posterior)
fitted_prediction = fitted_model.observation_mean(fitted_posterior)

In [ ]:
def plot_dynamics(true_model, initial_model, fitted_model):
    _, axs = plt.subplots(
        ncols=2,
        sharex='all',
        sharey='all',
        figsize=(8, 4),
    )

    plot.plot_dyn_linear_gaussian_comparison(
        true_model.dynamics,
        initial_model.dynamics,
        color1='xkcd:grey',
        ax=axs[0],
    )
    plot.plot_dyn_linear_gaussian_comparison(
        true_model.dynamics,
        fitted_model.dynamics,
        color1='xkcd:apple green',
        ax=axs[1],
    )


plot_dynamics(true_model, initial_model, fitted_model)

In [ ]:
def plot_observations_comparison(
    observations, initial_prediction, fitted_prediction, example_neuron
):
    _, axs = plt.subplots(
        nrows=4,
        sharex='all',
        constrained_layout=True,
        figsize=(8, 6),
        gridspec_kw={'height_ratios': [1, 1, 1, 2]},
    )

    ax = axs[0]

    plot.plot_traces_image(ax, observations)

    ax = axs[1]

    plot.plot_traces_image(ax, initial_prediction)

    ax = axs[2]

    plot.plot_traces_image(ax, fitted_prediction)

    ax = axs[3]
    ax.plot(
        observations[:, example_neuron],
        alpha=0.4,
        label=f'example neuron {example_neuron}',
        color='k',
    )
    ax.plot(
        initial_prediction[:, example_neuron], label='initial', color='xkcd:apple green'
    )
    ax.plot(fitted_prediction[:, example_neuron], label='fitted', color='xkcd:coral')
    ax.legend()


plot_observations_comparison(
    observations, initial_prediction, fitted_prediction, example_neuron=1
)